In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.neighbors import NearestNeighbors

In [2]:
df = pd.read_csv("homework_1.1.csv")

X_full = sm.add_constant(df[["X1", "X2", "X3"]])
model = sm.OLS(df["Y"], X_full).fit()
print(model.summary())

# simple vs. multivariate slope for each predictor
for col in ["X1", "X2", "X3"]:
    m = sm.OLS(df["Y"], sm.add_constant(df[[col]])).fit()
    print(f"{col}: simple={m.params[col]:.4f}  full={model.params[col]:.4f}  |gap|={abs(m.params[col] - model.params[col]):.4f}")

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                 3.543e+04
Date:                Sun, 21 Jun 2026   Prob (F-statistic):               0.00
Time:                        18:20:26   Log-Likelihood:                -727.62
No. Observations:                1000   AIC:                             1463.
Df Residuals:                     996   BIC:                             1483.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0026      0.016      0.166      0.8

In [3]:
df2 = pd.read_csv("homework_1.2.csv")

g0 = df2[df2["X"] == 0].reset_index(drop=True)
g1 = df2[df2["X"] == 1].reset_index(drop=True)

nn = NearestNeighbors(n_neighbors=1).fit(g0[["Z"]].values)
dist, idx = nn.kneighbors(g1[["Z"]].values)
dist, idx = dist.flatten(), idx.flatten()

print(f"max match distance: {dist.max():.6f}")
matched_Y0 = g0.iloc[idx]["Y"].values
print(f"effect (1-NN): {g1['Y'].mean() - matched_Y0.mean():.5f}")

nnr = NearestNeighbors(radius=0.2).fit(g0[["Z"]].values)
_, ridx = nnr.radius_neighbors(g1[["Z"]].values)

all_picked = np.concatenate([a for a in ridx if len(a) > 0])
duplicates = len(all_picked) - len(np.unique(all_picked))
print(f"duplicates: {duplicates}")

group_means = [g0.iloc[a]["Y"].mean() for a in ridx if len(a) > 0]
print(f"effect (radius 0.2): {g1['Y'].mean() - np.mean(group_means):.4f}")

max match distance: 0.210217
effect (1-NN): 0.54336
duplicates: 685
effect (radius 0.2): 0.5844
